# 05 — Circuit Tracing: Cross-Attention Head CompositionMechanistic interpretability analysis of the cross-attention circuit in PixArt-mini DiT.We trace how spatial-relationship information flows through the QK and OV circuitsand how heads compose across layers via the residual stream.**Extends NB02** (ablation + basic OV-QK ranking) with:- Full 36-head QK axis specialization analysis- Cross-layer read composition (W_q_downstream @ W_o_upstream)- Downstream reader feature selectivity

In [ ]:
# === CONFIG ===
import os, sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

RUN_NAME = "objrel_T5_DiT_mini_pilot"
CHECKPOINT_EPOCH = 4000
CHECKPOINT_STEP = 160000
N_PERM = 100
VP_FEATURES = ["spatial_relationship", "color1", "shape1", "color2", "shape2", "color1shape1", "color2shape2"]
N_SPATIAL_HEADS = 3
N_CONTROL_HEADS = 3
POS_EMBED_BASE_SIZE = 8
FIGDIR = os.path.join(PROJECT_ROOT, "Figures", "circuit_tracing")
os.makedirs(FIGDIR, exist_ok=True)

In [ ]:
import os, ssl, certifi, gc, time
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats

from utils.notebook_setup import (
    load_model_and_pipeline, load_embedding_cache,
    generate_prompts_and_scene_info, extract_projected_word_vectors,
    compute_head_alignment, select_spatial_and_control_heads,
    get_head_weights, compute_ov_matrix, compute_qk_matrix,
    transformer_hidden_size, transformer_head_dim,
    set_publication_style, saveallforms,
)

set_publication_style()

In [ ]:
# --- Load model and compute head alignment ---
transformer, pipe, tokenizer, device, compute_dtype = load_model_and_pipeline(
    RUN_NAME, CHECKPOINT_EPOCH, CHECKPOINT_STEP, PROJECT_ROOT
)

emb_cache = load_embedding_cache(RUN_NAME, PROJECT_ROOT)
prompts, scene_infos, scene_df = generate_prompts_and_scene_info()
_, wordvec_proj = extract_projected_word_vectors(
    emb_cache, transformer, tokenizer, prompts, scene_infos
)

align_df, effect_vecs, levels_map, r2_total, pos_embed_2d, ramp_templates = \
    compute_head_alignment(
        transformer, wordvec_proj, scene_df, VP_FEATURES,
        n_perm=N_PERM, base_size=POS_EMBED_BASE_SIZE,
    )

spatial_heads, control_heads = select_spatial_and_control_heads(
    align_df, N_SPATIAL_HEADS, N_CONTROL_HEADS
)

print(f"\nVariance partition R² = {r2_total:.3f}")
print(f"Spatial heads: {spatial_heads}")
print(f"Control heads: {control_heads}")
print()
print(align_df.sort_values("abs_cosine", ascending=False)
      .to_string(index=False, float_format="{:.3f}".format))

## Section B — QK Axis SpecializationFor each of the 36 cross-attention heads, we compute how strongly its QK circuitaligns with each of the 8 spatial ramp directions. The QK map is$\mathbf{p}^\top W_Q^\top W_K \mathbf{e}$ where $\mathbf{p}$ is the 2Dsinusoidal positional embedding and $\mathbf{e}$ is a spatial effect vector fromthe variance partition. For each (head, direction) pair we report the maximum|cosine similarity| across all spatial effect vectors.

In [ ]:
# --- B1: QK alignment heatmap (all 36 heads x 8 directions) ---
spatial_ev = effect_vecs["spatial_relationship"]  # (n_levels, 384)
spatial_levels = list(levels_map["spatial_relationship"])
ramp_names = list(ramp_templates.keys())

n_layers = len(transformer.transformer_blocks)
n_heads_per_layer = transformer.config.num_attention_heads
all_heads = [(li, hi) for li in range(n_layers) for hi in range(n_heads_per_layer)]

align_matrix = np.zeros((len(all_heads), len(ramp_names)))
for row, (li, hi) in enumerate(all_heads):
    W_q, W_k, _, _ = get_head_weights(transformer, li, hi)
    for col, rn in enumerate(ramp_names):
        ramp = ramp_templates[rn]
        best_c = 0
        for idx in range(len(spatial_ev)):
            ev_t = torch.tensor(spatial_ev[idx], dtype=torch.float32)
            k_proj = W_k @ ev_t
            qk_map = pos_embed_2d @ W_q.T @ k_proj
            qk_c = qk_map - qk_map.mean()
            c = abs(F.cosine_similarity(qk_c.unsqueeze(0), ramp.unsqueeze(0)).item())
            if c > best_c:
                best_c = c
        align_matrix[row, col] = best_c

# Plot heatmap
head_labels = [f"L{li}H{hi}" for li, hi in all_heads]

fig, ax = plt.subplots(figsize=(8, 12))
im = ax.imshow(align_matrix, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)

ax.set_xticks(range(len(ramp_names)))
ax.set_xticklabels([r.replace("_", "\n") for r in ramp_names], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(head_labels)))
ax.set_yticklabels(head_labels, fontsize=7)

# Annotate cells
for i in range(align_matrix.shape[0]):
    for j in range(align_matrix.shape[1]):
        val = align_matrix[i, j]
        color = "white" if val > 0.6 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=5, color=color)

# Add horizontal lines between layers
for li in range(1, n_layers):
    ax.axhline(y=li * n_heads_per_layer - 0.5, color="black", linewidth=1.0)

# Highlight spatial heads
for li, hi in spatial_heads:
    row_idx = li * n_heads_per_layer + hi
    ax.get_yticklabels()[row_idx].set_color("red")
    ax.get_yticklabels()[row_idx].set_fontweight("bold")

plt.colorbar(im, ax=ax, label="max |cosine| with ramp", shrink=0.6)
ax.set_title("QK-Ramp Alignment: All 36 Heads × 8 Directions")
ax.set_xlabel("Ramp Direction")
ax.set_ylabel("Head")
plt.tight_layout()
saveallforms(FIGDIR, "qk_alignment_heatmap_all_heads")
plt.show()

In [ ]:
# --- B2: Best QK preference maps for top-3 spatial heads ---
fig, axes = plt.subplots(N_SPATIAL_HEADS, 3, figsize=(9, 3 * N_SPATIAL_HEADS))
if N_SPATIAL_HEADS == 1:
    axes = axes[np.newaxis, :]

for row_i, (li, hi) in enumerate(spatial_heads):
    W_q, W_k, _, _ = get_head_weights(transformer, li, hi)

    # Find best (effect_vec, ramp) pair
    best_cos = 0
    best_qk_map = None
    best_ramp_name = None
    best_ramp = None
    for rn in ramp_names:
        ramp = ramp_templates[rn]
        for idx in range(len(spatial_ev)):
            ev_t = torch.tensor(spatial_ev[idx], dtype=torch.float32)
            k_proj = W_k @ ev_t
            qk_map = pos_embed_2d @ W_q.T @ k_proj
            qk_c = qk_map - qk_map.mean()
            c = abs(F.cosine_similarity(qk_c.unsqueeze(0), ramp.unsqueeze(0)).item())
            if c > best_cos:
                best_cos = c
                best_qk_map = qk_c.clone()
                best_ramp_name = rn
                best_ramp = ramp.clone()

    # Reshape to 2D grid
    qk_2d = best_qk_map.reshape(POS_EMBED_BASE_SIZE, POS_EMBED_BASE_SIZE).numpy()
    ramp_2d = best_ramp.reshape(POS_EMBED_BASE_SIZE, POS_EMBED_BASE_SIZE).numpy()

    vmax = max(abs(qk_2d.min()), abs(qk_2d.max()))
    ramp_vmax = max(abs(ramp_2d.min()), abs(ramp_2d.max()))

    # Col 0: ramp template
    axes[row_i, 0].imshow(ramp_2d, cmap="RdBu_r", vmin=-ramp_vmax, vmax=ramp_vmax,
                           interpolation="bicubic")
    axes[row_i, 0].set_title(f"Ramp: {best_ramp_name}", fontsize=9)
    axes[row_i, 0].set_ylabel(f"L{li}H{hi}", fontsize=10, fontweight="bold")

    # Col 1: QK map interpolated
    axes[row_i, 1].imshow(qk_2d, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                           interpolation="bicubic")
    axes[row_i, 1].set_title(f"QK map (bicubic)", fontsize=9)

    # Col 2: QK map raw 8x8
    im = axes[row_i, 2].imshow(qk_2d, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                                interpolation="nearest")
    axes[row_i, 2].set_title(f"QK map (raw) |cos|={best_cos:.3f}", fontsize=9)

    for c in range(3):
        axes[row_i, c].set_xticks([])
        axes[row_i, c].set_yticks([])

plt.suptitle("QK Spatial Preference Maps — Top Spatial Heads", fontsize=13, y=1.02)
plt.tight_layout()
saveallforms(FIGDIR, "qk_preference_maps_spatial_heads")
plt.show()

## Section C — Cross-Layer Read CompositionWhen an upstream head $h_1$ (layer $l_1$) writes to the residual stream via itsOV circuit ($W_O^{h_1}$), a downstream head $h_2$ (layer $l_2 > l_1$) reads fromthe residual via its Q projection ($W_Q^{h_2}$). The composition matrix$W_Q^{h_2} W_O^{h_1}$ quantifies how strongly $h_2$ reads $h_1$'s output.We measure this with the normalized Frobenius norm:$$\text{comp}(h_1 \to h_2) = \frac{\|W_Q^{h_2} W_O^{h_1}\|_F}{\|W_Q^{h_2}\|_F \cdot \|W_O^{h_1}\|_F}$$This is a purely weight-based (activation-free) measure of inter-head informationrouting, analogous to the Q-composition scores in Elhage et al. (2021).

In [ ]:
# --- C1: Compute composition scores ---
comp_scores = np.zeros((len(all_heads), len(all_heads)))
comp_scores[:] = np.nan  # default to NaN

for i, (l1, h1) in enumerate(all_heads):
    _, _, _, W_o_h1 = get_head_weights(transformer, l1, h1)  # (384, 64)
    for j, (l2, h2) in enumerate(all_heads):
        if l2 <= l1:
            continue  # can only read from earlier layer
        W_q_h2, _, _, _ = get_head_weights(transformer, l2, h2)  # (64, 384)
        composition = W_q_h2 @ W_o_h1  # (64, 64)
        comp_scores[i, j] = torch.linalg.norm(composition, "fro").item() / (
            torch.linalg.norm(W_q_h2, "fro").item() * torch.linalg.norm(W_o_h1, "fro").item() + 1e-12
        )

print(f"Composition matrix shape: {comp_scores.shape}")
print(f"Non-NaN entries: {np.count_nonzero(~np.isnan(comp_scores))}")
print(f"Score range: [{np.nanmin(comp_scores):.4f}, {np.nanmax(comp_scores):.4f}]")

In [ ]:
# --- C2: Composition heatmap ---
fig, ax = plt.subplots(figsize=(10, 10))

# Mask NaN for display
masked = np.ma.masked_invalid(comp_scores)
im = ax.imshow(masked, aspect="equal", cmap="viridis",
               vmin=np.nanmin(comp_scores), vmax=np.nanmax(comp_scores))

ax.set_xticks(range(len(head_labels)))
ax.set_xticklabels(head_labels, rotation=90, fontsize=6)
ax.set_yticks(range(len(head_labels)))
ax.set_yticklabels(head_labels, fontsize=6)

# Add layer boundary lines
for li in range(1, n_layers):
    pos = li * n_heads_per_layer - 0.5
    ax.axhline(y=pos, color="white", linewidth=0.5, alpha=0.5)
    ax.axvline(x=pos, color="white", linewidth=0.5, alpha=0.5)

# Highlight spatial head labels
spatial_indices = set()
for li, hi in spatial_heads:
    idx = li * n_heads_per_layer + hi
    spatial_indices.add(idx)
    ax.get_yticklabels()[idx].set_color("red")
    ax.get_yticklabels()[idx].set_fontweight("bold")
    ax.get_xticklabels()[idx].set_color("red")
    ax.get_xticklabels()[idx].set_fontweight("bold")

# Highlight control head labels
for li, hi in control_heads:
    idx = li * n_heads_per_layer + hi
    ax.get_yticklabels()[idx].set_color("blue")
    ax.get_xticklabels()[idx].set_color("blue")

plt.colorbar(im, ax=ax, label="Composition score (normalized Frobenius)", shrink=0.7)
ax.set_title("Cross-Layer Q-Composition: $W_Q^{h_2} W_O^{h_1}$\n"
             "(upstream writer → downstream reader)", fontsize=12)
ax.set_xlabel("Downstream reader $h_2$")
ax.set_ylabel("Upstream writer $h_1$")
plt.tight_layout()
saveallforms(FIGDIR, "cross_layer_composition_heatmap")
plt.show()

In [ ]:
# --- C3: Top compositions & spatial vs random comparison ---
# Extract all valid (upstream, downstream, score) triples
comp_triples = []
for i, (l1, h1) in enumerate(all_heads):
    for j, (l2, h2) in enumerate(all_heads):
        if not np.isnan(comp_scores[i, j]):
            comp_triples.append({
                "upstream": f"L{l1}H{h1}",
                "downstream": f"L{l2}H{h2}",
                "upstream_layer": l1, "upstream_head": h1,
                "downstream_layer": l2, "downstream_head": h2,
                "score": comp_scores[i, j],
            })

comp_df = pd.DataFrame(comp_triples)

# Tag spatial heads
spatial_set = set(spatial_heads)
comp_df["upstream_spatial"] = comp_df.apply(
    lambda r: (r.upstream_layer, r.upstream_head) in spatial_set, axis=1
)
comp_df["downstream_spatial"] = comp_df.apply(
    lambda r: (r.downstream_layer, r.downstream_head) in spatial_set, axis=1
)

# Top-10 compositions
print("=== Top-10 Cross-Layer Compositions ===")
top10 = comp_df.nlargest(10, "score")
print(top10[["upstream", "downstream", "score", "upstream_spatial", "downstream_spatial"]]
      .to_string(index=False, float_format="{:.4f}".format))

# Spatial vs non-spatial comparison
print("\n=== Spatial vs Non-Spatial Composition Scores ===")

# Upstream spatial vs non-spatial
up_sp = comp_df[comp_df.upstream_spatial]["score"]
up_ns = comp_df[~comp_df.upstream_spatial]["score"]
t_up, p_up = stats.ttest_ind(up_sp, up_ns, equal_var=False)
print(f"\nUpstream is spatial head:     mean={up_sp.mean():.4f} (n={len(up_sp)})")
print(f"Upstream is non-spatial head: mean={up_ns.mean():.4f} (n={len(up_ns)})")
print(f"  t={t_up:.3f}, p={p_up:.2e}")

# Downstream spatial vs non-spatial
dn_sp = comp_df[comp_df.downstream_spatial]["score"]
dn_ns = comp_df[~comp_df.downstream_spatial]["score"]
t_dn, p_dn = stats.ttest_ind(dn_sp, dn_ns, equal_var=False)
print(f"\nDownstream is spatial head:     mean={dn_sp.mean():.4f} (n={len(dn_sp)})")
print(f"Downstream is non-spatial head: mean={dn_ns.mean():.4f} (n={len(dn_ns)})")
print(f"  t={t_dn:.3f}, p={p_dn:.2e}")

# Which heads are the top readers of spatial heads?
spatial_writer_df = comp_df[comp_df.upstream_spatial]
top_readers = (spatial_writer_df.groupby(["downstream_layer", "downstream_head"])["score"]
              .mean().sort_values(ascending=False))
print("\n=== Top Readers of Spatial Heads (mean composition score) ===")
print(top_readers.head(10).to_string(float_format="{:.4f}".format))

## Section D — What Downstream Readers WriteFor the heads that show high Q-composition with spatial heads, we ask:what do they do with the spatial signal they read? We project each factor'seffect vectors through the OV circuit ($W_O W_V$) and measure thepreservation ratio $\|W_{OV} \mathbf{e}\| / \|\mathbf{e}\|$.If downstream readers selectively amplify spatial information (vs. color/shape),this supports a multi-layer spatial circuit rather than generic feature mixing.

In [ ]:
# --- D1: OV feature projection for spatial heads, top-readers, and controls ---

# Identify top-reader heads (highest mean composition with spatial upstream)
# Exclude spatial heads themselves since they are in L5 (no downstream)
top_reader_heads = []
for (dl, dh), _ in top_readers.head(6).items():
    if (dl, dh) not in spatial_set:
        top_reader_heads.append((dl, dh))
    if len(top_reader_heads) == 3:
        break

print(f"Top reader heads: {top_reader_heads}")
print(f"Spatial heads:    {spatial_heads}")
print(f"Control heads:    {control_heads}")

factors_to_test = ["spatial_relationship", "color1", "shape1"]
pres_rows = []

for label, heads in [("spatial", spatial_heads),
                     ("top-reader", top_reader_heads),
                     ("control", control_heads)]:
    for (li, hi) in heads:
        W_ov = compute_ov_matrix(transformer, li, hi)  # (384, 384)
        for factor in factors_to_test:
            ev = effect_vecs[factor]  # (n_levels, 384)
            ev_t = torch.tensor(ev, dtype=torch.float32)
            if ev_t.ndim == 1:
                ev_t = ev_t.unsqueeze(0)
            projected = W_ov @ ev_t.T  # (384, n_levels)
            proj_norm = projected.norm().item()
            orig_norm = ev_t.norm().item()
            preservation = proj_norm / (orig_norm + 1e-12)
            pres_rows.append({
                "head": f"L{li}H{hi}",
                "layer": li,
                "category": label,
                "factor": factor.replace("_", " "),
                "preservation": preservation,
            })

pres_df = pd.DataFrame(pres_rows)
print("\nOV Preservation ratios:")
pivot = pres_df.pivot_table(index="factor", columns="category",
                            values="preservation", aggfunc="mean")
print(pivot.to_string(float_format="{:.4f}".format))

In [ ]:
# --- D2: Feature selectivity bar chart ---
fig, ax = plt.subplots(figsize=(8, 5))

categories = ["spatial", "top-reader", "control"]
cat_colors = {"spatial": "#d62728", "top-reader": "#ff7f0e", "control": "#1f77b4"}
factors = sorted(pres_df["factor"].unique())
x = np.arange(len(factors))
width = 0.25

for i, cat in enumerate(categories):
    sub = pres_df[pres_df.category == cat]
    means = [sub[sub.factor == f]["preservation"].mean() for f in factors]
    sems = [sub[sub.factor == f]["preservation"].sem() for f in factors]
    ax.bar(x + i * width, means, width, yerr=sems, label=cat,
           color=cat_colors[cat], alpha=0.85, capsize=3)

ax.set_xticks(x + width)
ax.set_xticklabels(factors, fontsize=10)
ax.set_ylabel("OV Preservation Ratio $\\|W_{OV} \\mathbf{e}\\| / \\|\\mathbf{e}\\|$")
ax.set_title("Feature Selectivity of OV Circuits by Head Category")
ax.legend(title="Head category")
ax.set_ylim(bottom=0)
plt.tight_layout()
saveallforms(FIGDIR, "ov_feature_selectivity_bar")
plt.show()

# Print per-head detail
print("\nPer-head OV preservation detail:")
print(pres_df.pivot_table(index=["category", "head"], columns="factor",
                          values="preservation")
      .to_string(float_format="{:.4f}".format))

## Section E — Circuit SummaryThe table below summarizes the discovered cross-attention circuit. Each head'srole is determined by converging evidence from:- **QK alignment**: Does the head's attention pattern match a spatial ramp?- **Q-composition**: Does the head strongly read from (or write to) spatial heads?- **OV selectivity**: Does the head's output circuit preferentially preserve spatial vs. appearance features?| Component | Mechanism | Key Heads ||-----------|-----------|----------|| **Spatial QK routing** | QK circuit creates directional attention pattern from spatial effect vectors | Top-3 spatial heads (L5) || **Cross-layer composition** | Earlier-layer OV outputs are read by later-layer Q projections | Measured via $W_Q^{h_2} W_O^{h_1}$ Frobenius norm || **Feature-selective amplification** | OV circuits of different head categories show distinct factor preferences | Spatial heads vs. top-readers vs. controls |The circuit can be described as: spatial effect vectors in the projected textembeddings are routed to specific image positions via the QK circuit of spatialheads. These heads' OV outputs are then composed with downstream heads that mayfurther transform or integrate the spatial signal. The composition analysisreveals whether this is a dedicated spatial pathway or distributed processing.

In [ ]:
# --- E: Build summary table from actual results ---
summary_rows = []

# Spatial heads
for (li, hi) in spatial_heads:
    row_idx = li * n_heads_per_layer + hi
    best_dir = ramp_names[np.argmax(align_matrix[row_idx])]
    best_align = align_matrix[row_idx].max()
    # Get composition info (as upstream writer)
    head_idx = li * n_heads_per_layer + hi
    downstream_scores = comp_scores[head_idx, :]
    valid_ds = downstream_scores[~np.isnan(downstream_scores)]
    mean_ds = valid_ds.mean() if len(valid_ds) > 0 else 0
    summary_rows.append({
        "Head": f"L{li}H{hi}",
        "Layer": li,
        "Role": "Spatial router",
        "Best QK dir": best_dir,
        "|cos| QK-ramp": f"{best_align:.3f}",
        "Mean comp (as writer)": f"{mean_ds:.4f}" if len(valid_ds) > 0 else "N/A (last layer)",
    })

# Top reader heads
for (li, hi) in top_reader_heads:
    row_idx = li * n_heads_per_layer + hi
    best_dir = ramp_names[np.argmax(align_matrix[row_idx])]
    best_align = align_matrix[row_idx].max()
    # Composition as reader of spatial heads
    reader_idx = li * n_heads_per_layer + hi
    sp_writer_scores = []
    for (sl, sh) in spatial_heads:
        writer_idx = sl * n_heads_per_layer + sh
        s = comp_scores[writer_idx, reader_idx]
        if not np.isnan(s):
            sp_writer_scores.append(s)
    mean_read = np.mean(sp_writer_scores) if sp_writer_scores else 0
    summary_rows.append({
        "Head": f"L{li}H{hi}",
        "Layer": li,
        "Role": "Downstream reader",
        "Best QK dir": best_dir,
        "|cos| QK-ramp": f"{best_align:.3f}",
        "Mean comp (as writer)": f"reads spatial: {mean_read:.4f}",
    })

# Control heads
for (li, hi) in control_heads:
    row_idx = li * n_heads_per_layer + hi
    best_align = align_matrix[row_idx].max()
    summary_rows.append({
        "Head": f"L{li}H{hi}",
        "Layer": li,
        "Role": "Control (low spatial)",
        "Best QK dir": "-",
        "|cos| QK-ramp": f"{best_align:.3f}",
        "Mean comp (as writer)": "-",
    })

summary_df = pd.DataFrame(summary_rows)
print("=== Circuit Summary ===")
print(summary_df.to_string(index=False))
print(f"\nTotal heads analyzed: {len(all_heads)}")
print(f"Spatial heads identified: {len(spatial_heads)} (all in L5)")
print(f"Top downstream readers: {len(top_reader_heads)}")